# Matched vs Legacy Canary Sweep (sigma = 1.1, 3.0, 5.0)

**Goal:** replace the pathological high-sigma canary results (244%/657% tightness) with honest numbers from the
**matched (exchangeable) canary design**, and quantify the appearance artifact via the legacy-vs-matched delta.

**Runtime:** GPU required (T4 OK). Full sweep = 33 DP-SGD trainings x 15 epochs, roughly **2.5-4 h**.
Tip: run the smoke test first (2 min) to validate, then the full sweep.

**Steps:** (1) upload `matched_canary_bundle.zip`, (2) Run all, (3) download `matched_canary_results.zip` into `codex/results/`.

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload matched_canary_bundle.zip
import zipfile, os
zipfile.ZipFile('matched_canary_bundle.zip').extractall('project')
os.chdir('project')
print(sorted(os.listdir('.')))

In [ ]:
%pip -q install opacus dp-accounting pyyaml pytest
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY - fix runtime type!')

In [ ]:
# Sanity: the matched-design unit tests must pass before burning GPU hours.
!python -m pytest tests/test_matched_canaries.py -q

In [ ]:
# Smoke run (~2 min on GPU): 1 sigma, 2 seeds, 1 epoch, 100 canaries/group.
# Verifies the whole path end-to-end. Expect NO pathological rows.
!SMOKE=1 python codex/run_matched_canary_sweep.py

In [ ]:
# FULL sweep (~2.5-4 h). 3 sigmas x (1 base + 2 designs x 5 audit seeds) trainings, 15 epochs each.
!python codex/run_matched_canary_sweep.py

In [ ]:
import json, pandas as pd
rows = json.load(open('codex/results/matched_canary_sweep/matched_canary_summary.json'))
df = pd.DataFrame(rows)
cols = ['sigma','design','epsilon_lower_conservative','epsilon_lower_gdp','epsilon_upper_pld',
        'tightness_pld','pathological','member_score_mean','nonmember_score_mean','warning']
display(df[cols])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
fig, ax = plt.subplots(figsize=(7,4.5))
for design, marker in [('legacy','o'),('matched','s')]:
    sub = df[df.design==design].sort_values('sigma')
    ax.plot(sub.sigma, sub.epsilon_lower_conservative, marker+'-', label=f'{design} eps_lower')
sub = df[df.design=='matched'].sort_values('sigma')
ax.plot(sub.sigma, sub.epsilon_upper_pld, 'r:', label='eps_upper (PLD)')
ax.set_xlabel('noise multiplier sigma'); ax.set_ylabel('epsilon')
ax.set_title('Legacy vs matched canary design: the appearance artifact')
ax.legend(); plt.tight_layout()
plt.savefig('codex/results/matched_canary_sweep/legacy_vs_matched.png', dpi=150)
plt.show()

In [ ]:
!cd codex/results && zip -r ../../matched_canary_results.zip matched_canary_sweep
from google.colab import files
files.download('matched_canary_results.zip')